In [2]:
import pandas as pd

# 解决方案：使用正确的参数组合读取特殊格式文件
try:
    df = pd.read_csv(
        '/home/yuantao/code/LLM/txt/data/bioDBnet_db2db_cpdb.xls',
        sep='\t',                    # 明确指定制表符分隔
        header=0,                     # 使用文件第一行作为列名
        names=None,                   # 不要覆盖原始列名
        engine='python',              # 使用Python解析引擎处理复杂格式
        dtype={'Gene_Info': 'str'},   # 强制转换列类型
        encoding='ISO-8859-1',        # 尝试常见编码（可替换为utf-8/latin1等）
        quoting=3,                    # 处理可能存在的引号问题（QUOTE_NONE）
        on_bad_lines='warn',          # 遇到错误行时警告而不是报错
        skip_blank_lines=True         # 跳过空行
    )
    
    # 检查列名是否匹配
    if df.columns.tolist() != ['Ensembl_Gene_ID', 'Gene_Info']:
        print("检测到列名不匹配，自动修正列名...")
        df = df.rename(columns={
            df.columns[0]: 'Ensembl_Gene_ID',
            df.columns[1]: 'Gene_Info'
        })
except pd.errors.ParserError as e:
    print(f"解析错误：{e}")
    raise
def safe_extract_description(text):
    """安全解析Description信息的分步策略"""
    # 第一步：定位到[Description: 开头的标记
    desc_start = text.find('[Description: ')
    if desc_start == -1:
        return None
    
    # 第二步：定位对应的闭合括号
    balance = 1
    for i in range(desc_start + len('[Description: '), len(text)):
        if text[i] == '[':
            balance += 1
        elif text[i] == ']':
            balance -= 1
            if balance == 0:
                return text[desc_start+13:i].strip()  # 13是 '[Description: ' 的长度
    
    # 如果找不到闭合括号，取到末尾
    return text[desc_start+13:].strip()

# 应用解析函数
df['Description'] = df['Gene_Info'].apply(safe_extract_description)
cpdb_name = pd.read_csv('/home/yuantao/code/LLM/txt/data/node_names.txt', sep=',', header=None)
cpdb_name.columns = ['Ensembl_ID', 'Gene_Symbol']
cpdb_name = pd.merge(
    cpdb_name,
    df[['Ensembl_Gene_ID', 'Description']],
    how='left',  # 关键：左连接保留所有cpdb_name数据
    left_on='Ensembl_ID',
    right_on='Ensembl_Gene_ID',
    suffixes=('', '_from_df')
)
cpdb_name.to_csv('/home/yuantao/code/LLM/txt/data/cpdb_fullname+ens.csv', index=False)

检测到列名不匹配，自动修正列名...


In [3]:
# coding: gbk
from goatools.obo_parser import GODag
import pandas as pd
from goatools.base import download_go_basic_obo
import math
import numpy as np
# 下载并解析GO本体
obo_fname = download_go_basic_obo()
go_dag = GODag(obo_fname)

# 深度计算缓存字典
depth_cache = {}

def calculate_term_depth(term):
    """递归计算GO术语深度（带缓存优化）"""
    if term.item_id in depth_cache:
        return depth_cache[term.item_id]
    
    # 三大本体根节点判断
    if term.item_id in {'GO:0003674', 'GO:0005575', 'GO:0008150'}:
        depth_cache[term.item_id] = 0
        return 0
    
    # 获取所有父节点深度
    parent_depths = []
    for parent in term.parents:
        if parent.id in go_dag:
            parent_depths.append(calculate_term_depth(go_dag[parent.id]))
    
    # 计算当前深度
    current_depth = (max(parent_depths) if parent_depths else -1) + 1
    depth_cache[term.item_id] = current_depth
    return current_depth

def get_go_info(go_id):
    """获取GO术语名称和深度（集成处理）"""
    if pd.isna(go_id):
        return 'Unknown', 0
    
    go_id_clean = go_id.strip()
    if not go_id_clean.startswith('GO:'):
        return 'Unknown', 0
    
    term = go_dag.get(go_id_clean, None)
    if term:
        return term.name, calculate_term_depth(term)
    return 'Unknown', 0

# 读取GO注释数据
file_path = '/home/yuantao/code/LLM/GO词汇获取/25.H_sapiens.goa'
go_df = pd.read_csv(file_path, sep='\t', comment='!', header=None, low_memory=False)

columns = [
    "DB", "DB_Object_ID", "DB_Object_Symbol", "Qualifier", "GO_ID", 
    "DB_Reference", "Evidence_Code", "With_or_From", "Aspect", 
    "DB_Object_Name", "DB_Object_Synonym", "DB_Object_Type", 
    "Taxon", "Date", "Assigned_By", "Annotation_Extension", 
    "Gene_Product_Form_ID"
]
go_df.columns = columns

# 预处理核心数据
print("正在预处理数据...")
data = go_df[["DB_Object_ID", "DB_Object_Symbol", "Aspect", "DB_Object_Name", "GO_ID"]].copy()
data['DB_Object_Name'] = (
    data['DB_Object_Name']
    .astype(str)  # 强制转换为字符串类型
    .str.lower()  # 使用pandas字符串方法
    # .replace('nan', '')  # 将转换后的'nan'字符串替换为空值
)
data = data[~data['GO_ID'].isna()].drop_duplicates()
# print(data.head())
# 读取基因列表
print("正在读取基因列表...")
cpdb_name = pd.read_csv('/home/yuantao/code/LLM/txt/data/cpdb_fullname+ens.csv', sep=',', header=0)
cpdb_name.columns = ['Ensembl_ID', 'Gene_Symbol', 'Ensembl_Gene_ID','Description']


  EXISTS: go-basic.obo
go-basic.obo: fmt(1.2) rel(2024-10-27) 44,017 Terms
正在预处理数据...
正在读取基因列表...


In [8]:

# 合并数据
print("正在合并数据...")
# merged = pd.merge(cpdb_name, data, 
#                 left_on='Gene_Symbol', 
#                 right_on='DB_Object_Symbol', 
#                 how='left')
merged = pd.merge(cpdb_name, data, 
                left_on='Description', 
                right_on='DB_Object_Name', 
                how='left')
print(f"合并后的数据行数: {len(merged)}")
merged = merged.drop_duplicates(subset=['Ensembl_ID', 'GO_ID'])
print(f"去重后的数据行数: {len(merged)}")
# # 预计算所有唯一GO_ID的深度
# print("正在预先计算GO术语深度...")
# unique_go_ids = merged['GO_ID'].dropna().unique()
# for go_id in unique_go_ids:
#     get_go_info(go_id)

# # 添加GO术语信息和深度列
# merged[['GO_Term', 'Depth']] = merged['GO_ID'].apply(
#     lambda x: pd.Series(get_go_info(x))
# )
merged.head(20)

正在合并数据...
合并后的数据行数: 58881
去重后的数据行数: 44658


,Ensembl_ID,Gene_Symbol,Ensembl_Gene_ID,Description,DB_Object_ID,DB_Object_Symbol,Aspect,DB_Object_Name,GO_ID
0,ENSG00000167323,STIM1,ENSG00000167323,stromal interaction molecule 1,E9PR07,STIM1,F,stromal interaction molecule 1,GO:0005246
1,ENSG00000167323,STIM1,ENSG00000167323,stromal interaction molecule 1,E9PR07,STIM1,F,stromal interaction molecule 1,GO:0046872
2,ENSG00000167323,STIM1,ENSG00000167323,stromal interaction molecule 1,E9PR07,STIM1,C,stromal interaction molecule 1,GO:0016020
6,ENSG00000167323,STIM1,ENSG00000167323,stromal interaction molecule 1,Q13586,STIM1,F,stromal interaction molecule 1,GO:0002020
8,ENSG00000167323,STIM1,ENSG00000167323,stromal interaction molecule 1,Q13586,STIM1,F,stromal interaction molecule 1,GO:0005509
9,ENSG00000167323,STIM1,ENSG00000167323,stromal interaction molecule 1,Q13586,STIM1,F,stromal interaction molecule 1,GO:0005515
10,ENSG00000167323,STIM1,ENSG00000167323,stromal interaction molecule 1,Q13586,STIM1,F,stromal interaction molecule 1,GO:0042802
12,ENSG00000167323,STIM1,ENSG00000167323,stromal interaction molecule 1,Q13586,STIM1,F,stromal interaction molecule 1,GO:0051010
13,ENSG00000167323,STIM1,ENSG00000167323,stromal interaction molecule 1,Q13586,STIM1,P,stromal interaction molecule 1,GO:0002115
14,ENSG00000167323,STIM1,ENSG00000167323,stromal interaction molecule 1,Q13586,STIM1,P,stromal interaction molecule 1,GO:0005513


In [12]:
# # 过滤出现次数在100到200之间的GO术语（包含边界）
# print("正在过滤GO术语...")
go_id_counts = merged['GO_ID'].value_counts()  # 统计每个GO_ID出现次数
print(len(go_id_counts))
# valid_go_ids = go_id_counts[(go_id_counts >= 5) & (go_id_counts <= 200)].index  # 获取符合条件的GO_ID列表
valid_go_ids = go_id_counts[(go_id_counts >= 5)].index
print(f"有效GO_ID数量: {len(valid_go_ids)}")
merged = merged[merged['GO_ID'].isin(valid_go_ids)]  # 过滤数据
print(f"过滤后数据量: {len(merged)}")

1175
有效GO_ID数量: 1175
过滤后数据量: 24840


In [7]:

# 处理Aspect分类
print("正在处理Aspect分类...")
aspect_map = {
    'P': 'Biological Process',
    'C': 'Cellular Component',
    'F': 'Molecular Function'
}

def process_aspects(gene_df):
    # 展开多值Aspect
    gene_df['Aspect'] = gene_df['Aspect'].fillna('').str.split(', ')
    exploded = gene_df.explode('Aspect')
    
    # 过滤有效aspect
    valid_aspects = ['P', 'C', 'F']
    filtered = exploded[exploded['Aspect'].isin(valid_aspects)].copy()
    
    # 生成完整基因名称
    filtered['Full_Name'] = filtered.groupby(['Ensembl_ID', 'Gene_Symbol'])['DB_Object_Name'].transform(
        lambda x: x.dropna().iloc[0] if not x.dropna().empty else x.name[1]
    )
    
    # 去重：每个基因、Aspect、GO_ID保留最大深度
    deduped = filtered.groupby(
        ['Ensembl_ID', 'Gene_Symbol', 'Aspect', 'GO_ID'], 
        as_index=False
    ).agg({
        'Depth': 'max',
        'GO_Term': 'first',
        'Full_Name': 'first'
    })
    
    # 按深度降序排序
    sorted_deduped = deduped.sort_values('Depth', ascending=False)
    
    # 聚合每个基因的每个Aspect的top6最深GO术语
    grouped = sorted_deduped.groupby(['Ensembl_ID', 'Gene_Symbol', 'Aspect']).head(10)
    
    # 转换为所需的格式（列表中的字典）
    term_info = grouped.groupby(
        ['Ensembl_ID', 'Gene_Symbol', 'Aspect', 'Full_Name']
    ).apply(
        lambda x: x[['GO_Term', 'Depth']].to_dict('records')
    ).reset_index(name='Term_Info')
    
    return term_info

processed = process_aspects(merged)

# 转换数据格式为宽表
print("正在转换数据格式...")
pivot_df = processed.pivot_table(
    index=['Ensembl_ID', 'Gene_Symbol', 'Full_Name'],
    columns='Aspect',
    values='Term_Info',
    aggfunc='first'
).reset_index()

def generate_description_with_flag(row):
    """安全处理数组结构的增强版描述生成函数"""
    base = f"{row['Gene_Symbol']} ({row.get('Full_Name', row['Gene_Symbol'])})"
    aspects_info = []
    has_annotation = False
    
    aspect_map = {'P': 'Biological Process', 'C': 'Cellular Component', 'F': 'Molecular Function'}

    for aspect in ['P', 'C', 'F']:
        term_info = row.get(aspect)
        
        # 安全类型检查（处理数组和Series）
        if isinstance(term_info, (pd.Series, np.ndarray)):
            term_info = term_info.tolist()  # 转换为列表
            
        # 处理空值和无效类型
        if pd.api.types.is_list_like(term_info) and not term_info:  # 处理空列表
            continue
        elif not isinstance(term_info, list) or pd.isna(term_info).any():
            continue
            
        # 过滤有效条目
        valid_terms = []
        for t in term_info:
            if (
                isinstance(t, dict) 
                and 'Depth' in t 
                and isinstance(t['Depth'], (int, float))
            ):
                valid_terms.append(t)
                
        if not valid_terms:
            continue
            
        # 标记存在有效注释
        has_annotation = True
        
        # 构建描述片段
        aspect_name = aspect_map[aspect]
        sorted_terms = sorted(valid_terms, key=lambda x: x['Depth'], reverse=True)[:6]
        terms_str = ', '.join([f"{t['GO_Term']}" for t in sorted_terms])
        aspects_info.append(f"{aspect_name}: {terms_str}")

    # 构建最终描述
    description = (
        f"{base} - " + "; ".join(aspects_info) + "." 
        if aspects_info 
        else f"{base} has no functional annotations"
    )
    
    return description, has_annotation

# 应用增强版函数
print("生成All_Description和Has_Description列...")
pivot_df[['All_Description', 'Has_Description']] = pivot_df.apply(
    generate_description_with_flag, 
    axis=1,
    result_type='expand'
)

# 合并数据并处理标签
print("正在整合最终数据...")
final_df = pd.merge(
    cpdb_name[['Ensembl_ID', 'Gene_Symbol', 'Description']],
    pivot_df[['Ensembl_ID', 'Gene_Symbol', 'All_Description', 'Has_Description']],
    how='left',
    on=['Ensembl_ID', 'Gene_Symbol']
)

# 处理未匹配的情况
final_df['All_Description'] = final_df['All_Description'].fillna(
    final_df['Gene_Symbol'] + " just have full name " + final_df['Description']
)
final_df['Has_Description'] = final_df['Has_Description'].fillna(False)

# 类型转换确保布尔值
final_df['Has_Description'] = final_df['Has_Description'].astype(bool)

# 保存结果
final_df[['Ensembl_ID', 'Gene_Symbol', 'All_Description','Has_Description']].to_csv(
    '/home/yuantao/code/LLM/GO词汇获取/test.csv', 
    index=False,
    encoding='gbk'
)

print("处理完成！输出文件已保存为 gene_all_descriptions.csv")

正在处理Aspect分类...


/tmp/ipykernel_3662193/152939447.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  term_info = grouped.groupby(


正在转换数据格式...
生成All_Description和Has_Description列...
正在整合最终数据...
处理完成！输出文件已保存为 gene_all_descriptions.csv


/tmp/ipykernel_3662193/152939447.py:132: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  final_df['Has_Description'] = final_df['Has_Description'].fillna(False)


In [8]:
final_df['Has_Description'].describe()

count     13627
unique        2
top        True
freq      13357
Name: Has_Description, dtype: object